## lets look at a bigger model domain

In [2]:
import sys
from pathlib import Path
import pandas as pd
import xarray as xr
import numpy as np

sys.path.insert(0, "/jetfs/home/gbrack/MSC_Thesis/flexpart_tools/src")

from flexpart_tools.read_binary import read_namelists, read_gridini
from flexpart_tools.era5_io import read_met
from flexpart_tools.era5_vertical import calculate_geometry, average_to_flexpart_layers, geometric_altitude

In [3]:
run_dir = Path(
    "/jetfs/scratch/gbrack/flexpart_output/"
    "new_delhi_30d_24h_hourly_releases_bigger_domain"
)

met_dir = Path("/jetfs/shared-data/ECMWF/ERA5_glob_0.5deg_1h")

times_ini, times_fin, heights, lat, lon, age = read_namelists(run_dir)

release = 1
i = release - 1

print("Rückwärtsdauer [Tage]", age / 86400)
print("Anfangszeit [UTC]", times_ini[i])
print("Releasezeit [UTC]", times_fin[i])
print("FLEXPART-Schichten", len(heights) - 1)

Rückwärtsdauer [Tage] 30.0
Anfangszeit [UTC] 2015-04-25 00:00:00
Releasezeit [UTC] 2015-05-25 00:00:00
FLEXPART-Schichten 42


In [3]:
ds_ini = read_gridini(run_dir, times_ini, heights, lat, lon)
s_ini = ds_ini.sensitivity.isel(valid_time=1, drop=True)

weight_sum = s_ini.sum(skipna=False).item()
print("Sum of initial weights:", weight_sum)

Sum of initial weights: 0.9940591713037394


In [4]:
ds_ini

<xarray.Dataset> Size: 1GB
Dimensions:      (valid_time: 25, height: 42, latitude: 230, longitude: 719)
Coordinates:
  * longitude    (longitude) float64 6kB -179.0 -178.5 -178.0 ... 179.5 180.0
  * latitude     (latitude) float64 2kB -30.0 -29.5 -29.0 ... 83.5 84.0 84.5
  * height       (height) float64 336B 100.0 250.0 500.0 ... 3e+04 4e+04 5e+04
  * valid_time   (valid_time) datetime64[ns] 200B 2015-04-25 ... 2015-04-26
Data variables:
    sensitivity  (valid_time, height, latitude, longitude) float64 1GB 0.0 .....
Attributes:
    description:  Sensitivity to initial conditions

In [5]:
initial_time = pd.Timestamp(times_ini[i])
met_file = met_dir / initial_time.strftime("%Y/%m/EA%Y%m%d%H")
met_ini = read_met(initial_time, met_dir).sel(
    latitude=s_ini.latitude.values,
    longitude=s_ini.longitude.values
)

print(met_ini.sizes["hybrid"])
print(met_ini.sizes["hybrid_half"])

height_full, height_half = calculate_geometry(
    met_ini, met_ini.a_half.values, met_ini.b_half.values
)

137
138


In [6]:
met_ini

<xarray.Dataset> Size: 183MB
Dimensions:     (hybrid: 137, latitude: 230, longitude: 719, hybrid_half: 138)
Coordinates:
    time        datetime64[ns] 8B 2015-04-25
    step        timedelta64[ns] 8B 00:00:00
  * hybrid      (hybrid) float64 1kB 1.0 2.0 3.0 4.0 ... 134.0 135.0 136.0 137.0
  * latitude    (latitude) float64 2kB -30.0 -29.5 -29.0 ... 83.5 84.0 84.5
  * longitude   (longitude) float64 6kB -179.0 -178.5 -178.0 ... 179.5 180.0
    valid_time  datetime64[ns] 8B 2015-04-25
    number      int64 8B 0
Dimensions without coordinates: hybrid_half
Data variables:
    t           (hybrid, latitude, longitude) float32 91MB 191.3 191.3 ... 265.2
    q           (hybrid, latitude, longitude) float32 91MB 1.39e-06 ... 0.001776
    sp          (latitude, longitude) float32 661kB 1.02e+05 ... 1.013e+05
    z           (latitude, longitude) float32 661kB 1.198 0.6704 ... 4.377
    a_half      (hybrid_half) float64 1kB 0.0 2.0 3.102 4.666 ... 3.758 0.0 0.0
    b_half      (hybrid_half) float64 1kB 0.0 0.0 0.0 0.0 ... 0.995 0.9976 1.0
Attributes:
    GRIB_edition:            2
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-11-13T06:56 GRIB to CDM+CF via cfgrib-0.9.1...

In [7]:
cp, Lv, g = 1004.6, 2501000.0, 9.81

ground_altitude = geometric_altitude(
    met_ini.z.transpose("latitude", "longitude").values.astype("float64")
)

energy_fields = {
    "cpT": cp * met_ini.t.transpose("hybrid", "latitude", "longitude"),
    "Lvq": Lv * met_ini.q.transpose("hybrid", "latitude", "longitude"),
    "gh": g * (height_full + ground_altitude)
}

initial_raw = {}

for name, values in energy_fields.items():
    layer_mean = average_to_flexpart_layers(values, height_half, s_ini.height.values)
    field = s_ini.copy(data=layer_mean)
    initial_raw[name] = (s_ini * field).sum(skipna=False).item()

initial_raw["MSE"] = sum(initial_raw.values())

initial_budget = pd.Series(initial_raw, name="raw_Jkg").to_frame()
initial_budget["noramlized_Jkg"] = initial_budget["raw_Jkg"] / weight_sum

initial_budget

,raw_Jkg,noramlized_Jkg
cpT,260138.903662,261693.580394
Lvq,7416.903894,7461.229782
gh,53980.425162,54303.030162
MSE,321536.232719,323457.840339


In [8]:
weights = ds_ini.sensitivity.sum(
    dim=["height", "latitude", "longitude"],
    skipna=False
).to_series()

print(weights)

valid_time
2015-04-25 00:00:00    0.993808
2015-04-25 01:00:00    0.994059
2015-04-25 02:00:00    0.994337
2015-04-25 03:00:00    0.995118
2015-04-25 04:00:00    0.995028
2015-04-25 05:00:00    0.994730
2015-04-25 06:00:00    0.994278
2015-04-25 07:00:00    0.994310
2015-04-25 08:00:00    0.994080
2015-04-25 09:00:00    0.994168
2015-04-25 10:00:00    0.993757
2015-04-25 11:00:00    0.993902
2015-04-25 12:00:00    0.994659
2015-04-25 13:00:00    0.995328
2015-04-25 14:00:00    0.989384
2015-04-25 15:00:00    0.985585
2015-04-25 16:00:00    0.987495
2015-04-25 17:00:00    0.990667
2015-04-25 18:00:00    0.992713
2015-04-25 19:00:00    0.993820
2015-04-25 20:00:00    0.994598
2015-04-25 21:00:00    0.994642
2015-04-25 22:00:00    0.994104
2015-04-25 23:00:00    0.994420
2015-04-26 00:00:00    0.994099
Name: sensitivity, dtype: float64


In [9]:
partinit_files = sorted(run_dir.glob("partinit_*.nc"))
particle_files = sorted(run_dir.glob("partoutput_*.nc"))

print("Run", run_dir)
print("partinit-files", [p.name for p in partinit_files])
print("Number of partoutput-files", len(particle_files))

Run /jetfs/scratch/gbrack/flexpart_output/new_delhi_30d_24h_hourly_releases_bigger_domain
partinit-files ['partinit_20150526000000.nc']
Number of partoutput-files 32


In [10]:
release = 1
initial_time = pd.Timestamp(times_ini[release - 1])

with xr.open_dataset(partinit_files[0]) as partinit:
    particle_ids = partinit.particle.where(
        partinit.release == release, drop=True
    )

with xr.open_mfdataset(particle_files, combine="by_coords") as particles:
    positions = particles[["lon", "lat", "z"]].sel(
        particle=particle_ids,
        time=initial_time
    ).load()

print("Timestamp [UTC]", initial_time)
print("particles in the release", len(particle_ids))
positions

Timestamp [UTC] 2015-04-25 00:00:00
particles in the release 100000


<xarray.Dataset> Size: 2MB
Dimensions:   (particle: 100000)
Coordinates:
  * particle  (particle) int32 400kB 2400001 2400002 2400003 ... 2499999 2500000
    time      datetime64[ns] 8B 2015-04-25
Data variables:
    lon       (particle) float32 400kB 115.6 -81.84 98.79 ... -17.38 -149.5
    lat       (particle) float32 400kB 28.43 -4.73 15.49 ... 7.209 56.98 2.414
    z         (particle) float32 400kB 3.922e+03 1.214e+04 ... 1.513e+03
Attributes: (12/37)
    Conventions:            CF-1.6
    title:                  FLEXPART model output
    git:                    c1ef432 Thu Oct 29 05:46:26 2026 +0100
    source:                 Version 11.0 (2023-07-11) model output
    history:                2026-11-06 09:42 +0100  created by gbrack on jet1...
    references:             Stohl et al., Atmos. Chem. Phys., 2005, doi:10.51...
    ...                     ...
    ind_source:             1
    ind_receptor:           2
    mquasilag:              0
    nested_output:          2
    sfc_only:               0
    linit_cond:             2

In [12]:
dx = abs(lon[1] - lon[0])
dy = abs(lat[1] - lat[0])

west, east = min(lon) - dx/2, max(lon) + dx/2
south, north = min(lat) - dy/2, max(lat) + dy/2

particle_lon = ((positions.lon + 180) % 360) - 180
particle_lat = positions.lat
valid = np.isfinite(particle_lon) & np.isfinite(particle_lat)

outside = {
    "longitude": ((particle_lon - west) % 360) >= (east - west),
    "southern": particle_lat < south,
    "northern": particle_lat >= north,
}

outside_any = valid & (
    outside["longitude"] | outside["southern"] | outside["northern"]
)
n_valid = int(valid.sum().item())
n_outside = int(outside_any.sum().item())

print(f"Outside: {n_outside:,} / {n_valid:,} valid positions "
      f"({n_outside / n_valid:.2%})")

for direction, mask in outside.items():
    print(direction, int((valid & mask).sum().item()))

Outside: 619 / 100,000 valid positions (0.62%)
longitude 150
southern 171
northern 299


In [13]:
release = 1
s_ini = ds_ini.sensitivity.isel(valid_time=release - 1, drop=True)

inside_fraction = 1 - n_outside / n_valid
weight_sum = s_ini.sum(skipna=False).item()

print(f"Partikel innerhalb:       {inside_fraction:.6f}")
print(f"Summe aus grid_initial:   {weight_sum:.6f}")
print(f"Differenz in Prozentpunkten: {100 * (weight_sum - inside_fraction):+.4f}")

Partikel innerhalb:       0.993810
Summe aus grid_initial:   0.993808
Differenz in Prozentpunkten: -0.0002
